# RoadGuard AI — Final Evaluation

Clean reproducibility notebook for the final RoadGuard model.

This notebook contains only the final evaluation workflow:
1. Environment and paths
2. RDD2022 validation/test extraction
3. Improved-model validation
4. FP/FN analysis
5. Final held-out test
6. Efficiency measurement
7. Saudi external-domain prediction check
8. Export of report-ready tables and figures

> The held-out test set is used only after model selection.
> The Saudi dataset is not used for training and is treated as an external-domain check.


## 1. Setup


In [ ]:
!pip install -q ultralytics

from pathlib import Path
from ultralytics import YOLO
import zipfile
import csv
import shutil
import time
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

MODEL_PATH = Path(
    "/kaggle/input/datasets/adnanbaothman/"
    "roadguard-final-evaluation-files/roadguard_best.pt"
)

RDD_ZIP = Path(
    "/kaggle/input/notebooks/adnanbaothman/"
    "notebook225b511d78/rdd2022_yolo.zip"
)

RDD_ROOT = Path("/kaggle/working/rdd2022_yolo")
FINAL_EVAL = Path("/kaggle/working/final_evaluation")
FINAL_EVAL.mkdir(parents=True, exist_ok=True)

print("Model exists:", MODEL_PATH.exists())
print("RDD ZIP exists:", RDD_ZIP.exists())


## 2. Extract validation and test splits


In [ ]:
wanted_prefixes = (
    "rdd2022_yolo/val/images/",
    "rdd2022_yolo/val/labels/",
    "rdd2022_yolo/test/images/",
    "rdd2022_yolo/test/labels/",
)

wanted_files = {"rdd2022_yolo/data.yaml"}

with zipfile.ZipFile(RDD_ZIP, "r") as z:
    members = [
        name for name in z.namelist()
        if any(name.startswith(prefix) for prefix in wanted_prefixes)
        or name in wanted_files
    ]
    print("Files selected:", len(members))
    z.extractall("/kaggle/working", members)

print("Val images:", len(list((RDD_ROOT / "val/images").glob("*"))))
print("Val labels:", len(list((RDD_ROOT / "val/labels").glob("*.txt"))))
print("Test images:", len(list((RDD_ROOT / "test/images").glob("*"))))
print("Test labels:", len(list((RDD_ROOT / "test/labels").glob("*.txt"))))


## 3. Create evaluation YAML


In [ ]:
DATA_YAML = RDD_ROOT / "data_eval.yaml"

DATA_YAML.write_text(
    '''path: /kaggle/working/rdd2022_yolo
train: train/images
val: val/images
test: test/images
names:
  0: D00
  1: D10
  2: D20
  3: D40
''',
    encoding="utf-8"
)

print(DATA_YAML.read_text())


## 4. Final / improved model validation


In [ ]:
model = YOLO(str(MODEL_PATH))

val_metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=32,
    device=0,
    plots=True,
    save_json=False,
    project=str(FINAL_EVAL),
    name="improved_validation",
    exist_ok=True
)

print("\n=== VALIDATION OVERALL ===")
print("Precision:", val_metrics.box.mp)
print("Recall:", val_metrics.box.mr)
print("mAP50:", val_metrics.box.map50)
print("mAP50-95:", val_metrics.box.map)

print("\n=== VALIDATION PER CLASS ===")
for i, name in model.names.items():
    print(
        name,
        "P:", val_metrics.box.p[i],
        "R:", val_metrics.box.r[i],
        "AP50:", val_metrics.box.ap50[i],
        "AP50-95:", val_metrics.box.maps[i],
    )

print("\nPlots saved to:", val_metrics.save_dir)


## 5. Generate operational-threshold predictions for FP/FN analysis


In [ ]:
PRED_DIR = FINAL_EVAL / "improved_predictions"

for _ in model.predict(
    source=str(RDD_ROOT / "val/images"),
    conf=0.26,
    imgsz=640,
    device=0,
    save=False,
    save_txt=True,
    save_conf=True,
    project=str(FINAL_EVAL),
    name="improved_predictions",
    exist_ok=True,
    verbose=False,
    stream=True,
):
    pass

print("Prediction labels:", len(list((PRED_DIR / "labels").glob("*.txt"))))


## 6. FP/FN analysis

Custom one-to-one greedy matching at:
- confidence = 0.26
- IoU = 0.50

Misclassified matches are counted separately from unmatched FP/FN.


In [ ]:
from collections import defaultdict

GT_DIR = RDD_ROOT / "val/labels"
PRED_LABEL_DIR = PRED_DIR / "labels"
OUTPUT_DIR = FINAL_EVAL / "improved_error_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IOU_THRESHOLD = 0.50
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}

def read_labels(path, prediction=False):
    boxes = []
    if not path.exists():
        return boxes

    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls_id = int(float(parts[0]))
            xc, yc, w, h = map(float, parts[1:5])
            boxes.append({
                "class_id": cls_id,
                "box": (
                    xc - w / 2,
                    yc - h / 2,
                    xc + w / 2,
                    yc + h / 2,
                ),
                "confidence": float(parts[5]) if prediction and len(parts) >= 6 else None,
            })
    return boxes

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    a2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = a1 + a2 - inter

    return 0 if union == 0 else inter / union

summary = defaultdict(lambda: {"TP": 0, "FP": 0, "FN": 0, "Misclassification": 0})
details = []

gt_files = sorted(GT_DIR.glob("*.txt"))

for index, gt_path in enumerate(gt_files, start=1):
    pred_path = PRED_LABEL_DIR / gt_path.name
    gt_boxes = read_labels(gt_path)
    pred_boxes = read_labels(pred_path, prediction=True)

    candidates = []
    for gi, gt in enumerate(gt_boxes):
        for pi, pred in enumerate(pred_boxes):
            score = iou(gt["box"], pred["box"])
            if score >= IOU_THRESHOLD:
                candidates.append((score, gi, pi))

    candidates.sort(reverse=True)
    matched_gt, matched_pred = set(), set()

    for score, gi, pi in candidates:
        if gi in matched_gt or pi in matched_pred:
            continue

        matched_gt.add(gi)
        matched_pred.add(pi)

        gt = gt_boxes[gi]
        pred = pred_boxes[pi]
        true_class = CLASS_NAMES[gt["class_id"]]
        pred_class = CLASS_NAMES[pred["class_id"]]

        if gt["class_id"] == pred["class_id"]:
            summary[true_class]["TP"] += 1
            error_type = "TP"
        else:
            summary[true_class]["Misclassification"] += 1
            error_type = "Misclassification"

        details.append({
            "image": gt_path.stem,
            "true_class": true_class,
            "predicted_class": pred_class,
            "confidence": pred["confidence"],
            "iou": round(score, 4),
            "error_type": error_type,
        })

    for gi, gt in enumerate(gt_boxes):
        if gi not in matched_gt:
            true_class = CLASS_NAMES[gt["class_id"]]
            summary[true_class]["FN"] += 1
            details.append({
                "image": gt_path.stem,
                "true_class": true_class,
                "predicted_class": "None",
                "confidence": "",
                "iou": "",
                "error_type": "FN",
            })

    for pi, pred in enumerate(pred_boxes):
        if pi not in matched_pred:
            pred_class = CLASS_NAMES[pred["class_id"]]
            summary[pred_class]["FP"] += 1
            details.append({
                "image": gt_path.stem,
                "true_class": "None",
                "predicted_class": pred_class,
                "confidence": pred["confidence"],
                "iou": "",
                "error_type": "FP",
            })

    if index % 500 == 0:
        print(f"Processed {index}/{len(gt_files)}")

summary_rows = []
for class_name in ["D00", "D10", "D20", "D40"]:
    values = summary[class_name]
    summary_rows.append({
        "Class": class_name,
        "TP": values["TP"],
        "FP": values["FP"],
        "FN": values["FN"],
        "Misclassification": values["Misclassification"],
    })

pd.DataFrame(summary_rows).to_csv(
    OUTPUT_DIR / "improved_fp_fn_summary.csv", index=False
)
pd.DataFrame(details).to_csv(
    OUTPUT_DIR / "improved_fp_fn_results.csv", index=False
)

display(pd.DataFrame(summary_rows))


## 7. Final held-out test


In [ ]:
test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=32,
    device=0,
    plots=True,
    save_json=False,
    project=str(FINAL_EVAL),
    name="final_test",
    exist_ok=True
)

print("\n=== FINAL TEST OVERALL ===")
print("Precision:", test_metrics.box.mp)
print("Recall:", test_metrics.box.mr)
print("mAP50:", test_metrics.box.map50)
print("mAP50-95:", test_metrics.box.map)

print("\n=== FINAL TEST PER CLASS ===")
for i, name in model.names.items():
    print(
        name,
        "P:", test_metrics.box.p[i],
        "R:", test_metrics.box.r[i],
        "AP50:", test_metrics.box.ap50[i],
        "AP50-95:", test_metrics.box.maps[i],
    )

print("\nPlots saved to:", test_metrics.save_dir)


## 8. Final model efficiency


In [ ]:
VAL_IMAGES = RDD_ROOT / "val/images"

model_size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)
total_params = sum(p.numel() for p in model.model.parameters())

images = sorted([
    p for p in VAL_IMAGES.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

test_images = images[:100]

for img in test_images[:10]:
    _ = model.predict(str(img), imgsz=640, device=0, verbose=False)

torch.cuda.synchronize()

latencies = []

for img in test_images:
    torch.cuda.synchronize()
    start = time.perf_counter()

    _ = model.predict(str(img), imgsz=640, device=0, verbose=False)

    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - start) * 1000)

mean_latency = float(np.mean(latencies))
median_latency = float(np.median(latencies))
std_latency = float(np.std(latencies))
fps = 1000 / mean_latency

print("Model size:", round(model_size_mb, 2), "MB")
print("Parameters:", f"{total_params:,}")
print("Mean latency:", round(mean_latency, 2), "ms/image")
print("Median latency:", round(median_latency, 2), "ms/image")
print("Latency std:", round(std_latency, 2), "ms")
print("Approx throughput:", round(fps, 1), "images/sec")
print("Device:", torch.cuda.get_device_name(0))


## 9. Saudi external-domain prediction check

This is a **prediction-presence / qualitative external-domain check**, not a directly comparable Saudi mAP evaluation.

The Saudi dataset uses a different class/annotation scheme from RDD2022.
Current Saudi results are preliminary and should be reviewed separately.


In [ ]:
SAUDI_ROOT = Path(
    "/kaggle/input/datasets/adnanbaothman/"
    "saudi-pothole-full-evaluation"
)

SAUDI_IMAGES = SAUDI_ROOT / "images"
SAUDI_OUT = FINAL_EVAL / "saudi_predictions_sorted"
DETECTED_DIR = SAUDI_OUT / "detected_with_boxes"
NO_DETECTION_DIR = SAUDI_OUT / "no_detection"

DETECTED_DIR.mkdir(parents=True, exist_ok=True)
NO_DETECTION_DIR.mkdir(parents=True, exist_ok=True)

saudi_images = sorted([
    p for p in SAUDI_IMAGES.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

rows = []
detected_count = 0
no_detection_count = 0

for i, image_path in enumerate(saudi_images, start=1):
    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.26,
        device=0,
        verbose=False
    )[0]

    if result.boxes is not None and len(result.boxes) > 0:
        annotated = result.plot()
        cv2.imwrite(str(DETECTED_DIR / image_path.name), annotated)

        detections = []
        for cls, conf in zip(
            result.boxes.cls.cpu().numpy(),
            result.boxes.conf.cpu().numpy()
        ):
            cls = int(cls)
            detections.append(f"{model.names[cls]} {float(conf):.2f}")

        rows.append({
            "image": image_path.name,
            "status": "Detected",
            "detections": ", ".join(detections),
        })
        detected_count += 1
    else:
        shutil.copy2(image_path, NO_DETECTION_DIR / image_path.name)
        rows.append({
            "image": image_path.name,
            "status": "No detection",
            "detections": "",
        })
        no_detection_count += 1

    if i % 500 == 0:
        print(f"Processed {i}/{len(saudi_images)}")

saudi_df = pd.DataFrame(rows)
saudi_df.to_csv(SAUDI_OUT / "saudi_prediction_summary.csv", index=False)

print("Saudi images:", len(saudi_images))
print("Detected:", detected_count)
print("No detection:", no_detection_count)
print("Prediction presence rate:", round(detected_count / len(saudi_images) * 100, 2), "%")


## 10. Report-ready summary tables and figures


In [ ]:
PACKAGE = Path("/kaggle/working/roadguard_final_package")
TABLES = PACKAGE / "tables"
FIGURES = PACKAGE / "figures"
IMPROVED_FIGURES = FIGURES / "improved_model"

for p in [TABLES, FIGURES, IMPROVED_FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

overall_val = pd.DataFrame([
    {"Model": "Baseline", "Precision": 0.668, "Recall": 0.548, "mAP50": 0.603, "mAP50-95": 0.317},
    {"Model": "Improved D40 Oversampling", "Precision": 0.6639095849, "Recall": 0.5523327957, "mAP50": 0.6083924743, "mAP50-95": 0.3217212405},
])
overall_val.to_csv(TABLES / "validation_overall_baseline_vs_improved.csv", index=False)

per_class_val = pd.DataFrame([
    ["Baseline", "D00", 0.643, 0.546, 0.586, 0.339],
    ["Baseline", "D10", 0.636, 0.563, 0.597, 0.302],
    ["Baseline", "D20", 0.705, 0.643, 0.703, 0.388],
    ["Baseline", "D40", 0.687, 0.439, 0.525, 0.240],
    ["Improved", "D00", 0.6661805487, 0.5339872855, 0.5880628091, 0.3385616985],
    ["Improved", "D10", 0.6660744157, 0.5400148566, 0.5980149076, 0.2999345731],
    ["Improved", "D20", 0.7153155280, 0.6446748351, 0.7149568751, 0.3969274310],
    ["Improved", "D40", 0.6080678472, 0.4906542056, 0.5325353054, 0.2514612595],
], columns=["Model", "Class", "Precision", "Recall", "AP50", "AP50-95"])
per_class_val.to_csv(TABLES / "validation_per_class_baseline_vs_improved.csv", index=False)

test_results = pd.DataFrame([
    ["ALL", 3845, 5550, 0.6477019994, 0.5419418737, 0.5929187264, 0.3048487268],
    ["D00", 1356, 2656, 0.6529350777, 0.5404503221, 0.5834288501, 0.3219142934],
    ["D10", 770, 1144, 0.6476926485, 0.5576923077, 0.6054647068, 0.3073444113],
    ["D20", 850, 1091, 0.7019773560, 0.6122823098, 0.6963860046, 0.3703428396],
    ["D40", 373, 659, 0.5882029155, 0.4573425554, 0.4863953442, 0.2197933630],
], columns=["Class", "Images", "Instances", "Precision", "Recall", "AP50", "AP50-95"])
test_results.to_csv(TABLES / "final_test_metrics.csv", index=False)

efficiency = pd.DataFrame([{
    "Model": "RoadGuard Final YOLO11s",
    "Model Size MB": 18.29,
    "Parameters": 9429340,
    "GFLOPs": 21.4,
    "Image Size": 640,
    "Device": "Tesla T4",
    "Ultralytics Test Inference ms/image": 9.5,
    "Independent Mean Latency ms/image": 12.71,
    "Independent Median Latency ms/image": 12.56,
    "Latency Std ms": 1.01,
    "Approx FPS": 78.7,
}])
efficiency.to_csv(TABLES / "model_efficiency.csv", index=False)

# Copy official improved-model plots generated by Ultralytics
src = FINAL_EVAL / "improved_validation"
for name in [
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
]:
    if (src / name).exists():
        shutil.copy2(src / name, IMPROVED_FIGURES / name)

print("Report package:", PACKAGE)


## 11. Create final ZIP


In [ ]:
ZIP_BASE = Path("/kaggle/working/roadguard_final_github_report_package")

shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    PACKAGE
)

print("Created:", str(ZIP_BASE) + ".zip")
